# Análisis Exploratorio de Datos (EDA): Correlación Clima - Casos de Dengue

Este cuaderno realiza un análisis exploratorio de datos para estudiar la relación entre las variables climatológicas (temperatura máxima y precipitación acumulada) y la incidencia de casos de dengue registrados en la provincia de Santa Elena.

## Objetivos
1. Cargar y explorar los datos limpios en la capa de Staging (`.parquet`).
2. Analizar la distribución de la temperatura y la precipitación.
3. Realizar un análisis de correlación con desfase (Lag Analysis) para comprobar si la lluvia acumulada correlaciona con un brote de casos de dengue semanas después (desfase temporal de 1 a 6 semanas).

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Estilo de gráficos premium
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["font.size"] = 11

## 1. Carga de Datos desde la Zona Staging

In [ ]:
# Rutas relativas desde el directorio docs/
ruta_clima = "../staging/stg_clima.parquet"
ruta_casos = "../staging/stg_casos_dengue.parquet"

if os.path.exists(ruta_clima) and os.path.exists(ruta_casos):
    df_clima = pd.read_parquet(ruta_clima)
    df_casos = pd.read_parquet(ruta_casos)
    print(f"[ÉXITO] Clima cargado: {df_clima.shape} registros")
    print(f"[ÉXITO] Casos cargados: {df_casos.shape} registros")
else:
    print("[ERROR] No se encontraron los archivos parquet en la carpeta staging. Por favor ejecuta el pipeline primero.")

## 2. Inspección Inicial y Unión (Merge)

In [ ]:
# Renombrar columna de semana en clima para que coincida con casos si es necesario
if 'semana' in df_clima.columns:
    df_clima = df_clima.rename(columns={'semana': 'semana_epidemiologica'})

# Unir los conjuntos de datos
df_merge = pd.merge(
    df_casos,
    df_clima,
    on=['canton', 'anio', 'semana_epidemiologica'],
    how='inner'
)
print(f"Datos unificados: {df_merge.shape} registros")
df_merge.head()

## 3. Análisis de Correlación Cruzada con Desfase (Lag Correlation Analysis)
Dado que los mosquitos requieren tiempo para reproducirse tras lluvias intensas, evaluamos la correlación entre la precipitación en la semana $t - d$ y los casos en la semana $t$.

In [ ]:
# Crear un dataframe ordenado temporalmente por cantón para calcular lags
df_sorted = df_merge.sort_values(by=['canton', 'anio', 'semana_epidemiologica']).copy()

# Calcular correlaciones para desfases (lags) de 0 a 6 semanas
correlaciones = []
lags = list(range(7))

for lag in lags:
    # Crear columna con desplazamiento temporal (lag)
    df_sorted[f'precipitacion_lag_{lag}'] = df_sorted.groupby('canton')['precipitacion_mm'].shift(lag)
    
    # Calcular correlación de Pearson
    corr = df_sorted['casos_confirmados'].corr(df_sorted[f'precipitacion_lag_{lag}'])
    correlaciones.append(corr)
    print(f"Correlación con Desfase de {lag} semanas: {corr:.4f}")

# Graficar los coeficientes de correlación
plt.figure(figsize=(10, 5))
plt.plot(lags, correlaciones, marker='o', color='#053A90', linewidth=2, markersize=8)
plt.title('Correlación entre Lluvia y Casos de Dengue según Semanas de Desfase (Lag Analysis)', fontsize=14, fontweight='bold', pad=15)
plt.xlabel('Semanas de Desfase (Precipitación en t - d)', fontsize=12)
plt.ylabel('Coeficiente de Correlación (Pearson)', fontsize=12)
plt.xticks(lags)
plt.axvline(x=np.argmax(correlaciones), color='#057E3F', linestyle='--', label=f'Correlación Máxima (Lag={np.argmax(correlaciones)} sem)')
plt.legend(frameon=True, facecolor='white')
plt.tight_layout()
plt.show()

## 4. Visualización de la Serie Temporal en Santa Elena
Graficamos la evolución temporal cruzada de lluvia y nuevos casos de dengue.

In [ ]:
# Filtrar para un cantón crítico, por ejemplo SANTA ELENA
df_se = df_sorted[df_sorted['canton'] == 'SANTA ELENA'].copy()

if not df_se.empty:
    fig, ax1 = plt.subplots(figsize=(12, 6))

    # Eje 1: Casos
    color = '#053A90'
    ax1.set_xlabel('Semana Epidemiológica')
    ax1.set_ylabel('Casos Confirmados', color=color, fontweight='bold')
    line1 = ax1.plot(df_se['semana_epidemiologica'].astype(str) + "/" + df_se['anio'].astype(str), df_se['casos_confirmados'], color=color, marker='o', label='Casos Dengue', linewidth=2)
    ax1.tick_params(axis='y', labelcolor=color)

    # Eje 2: Precipitación
    ax2 = ax1.twinx()
    color = '#4798E4'
    ax2.set_ylabel('Precipitación (mm)', color=color, fontweight='bold')
    line2 = ax2.plot(df_se['semana_epidemiologica'].astype(str) + "/" + df_se['anio'].astype(str), df_se['precipitacion_mm'], color=color, linestyle='--', marker='x', label='Lluvia (mm)')
    ax2.tick_params(axis='y', labelcolor=color)

    # Agregar etiquetas y leyendas
    lines = line1 + line2
    labels = [l.get_label() for l in lines]
    ax1.legend(lines, labels, loc='upper left')
    
    plt.title('Relación Temporal: Lluvia vs Casos en Santa Elena', fontsize=14, fontweight='bold', pad=15)
    # Ajustar frecuencia de etiquetas en eje X para legibilidad
    ticks = ax1.get_xticks()
    ax1.set_xticks(ticks[::5])
    
    plt.tight_layout()
    plt.show()
else:
    print("No hay suficientes datos del cantón SANTA ELENA para graficar.")